In [1]:
import numpy as np
import pandas as pd

from utils.data import DataManager
from utils.analysis.risk_metrics.analyzers import (
    VarEsAnalyzer,
    RatioAnalyzer,
    DistributionAnalyzer
)
from utils.tools import (
    BENCHMARKS,
    RISK_ANALYSIS,
    INTERPRETATION_THRESHOLDS
)

In [ ]:
TICKERS = ["META"] 
BENCHMARK = "SP500" 
START_DATE = "2020-01-01" 
END_DATE = "2024-12-31"
RISK_FREE_RATE = RISK_ANALYSIS['risk_free_rate']
ANNUAL_FACTOR = RISK_ANALYSIS['annual_factor']
DEFAULT_CONFIDENCE = RISK_ANALYSIS['default_confidence_level']
CONFIDENCE_LEVELS = RISK_ANALYSIS['default_confidence_levels']
ROLLING_WINDOW = RISK_ANALYSIS['rolling']['default_window']
MC_SIMULATIONS = RISK_ANALYSIS['monte_carlo']['n_simulations']
MC_SEED = RISK_ANALYSIS['monte_carlo']['seed']
WEIGHTS = np.ones(len(TICKERS)) / len(TICKERS)
SKEW_THRESH = INTERPRETATION_THRESHOLDS['skewness']
KURT_THRESH = INTERPRETATION_THRESHOLDS['kurtosis']

In [3]:
data_manager = DataManager()
var_es_analyzer = VarEsAnalyzer(annual_factor=ANNUAL_FACTOR)
ratio_analyzer = RatioAnalyzer(annual_factor=ANNUAL_FACTOR)
dist_analyzer = DistributionAnalyzer()

In [ ]:
assets_prices, benchmark_prices = data_manager.download_portfolio_with_benchmark(
    tickers=TICKERS,
    benchmark_name=BENCHMARK,
    start_date=START_DATE,
    end_date=END_DATE
)

returns = assets_prices.pct_change().dropna()
benchmark_returns = benchmark_prices.pct_change().dropna()

Descargando portafolio completo...
Período: 2020-01-01 → 2024-12-31


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Período: 2020-01-01 → 2024-12-31
Portafolio descargado: 1 activos + benchmark


In [5]:
all_ratios = ratio_analyzer.calculate_all_ratios(
    returns, 
    WEIGHTS, 
    RISK_FREE_RATE 
)

df_ratios = pd.DataFrame({
    'Valor Formateado': [
        f"{all_ratios['sharpe_ratio']:.3f}",
        f"{all_ratios['sortino_ratio']:.3f}",
        f"{all_ratios['annual_return']*100:.2f}%",
        f"{all_ratios['annual_volatility']*100:.2f}%",
        f"{all_ratios['downside_volatility']*100:.2f}%",
        f"{all_ratios['excess_return']*100:.2f}%",
        f"{all_ratios['risk_free_rate']*100:.2f}%"
    ]
}, index=[
    'Sharpe Ratio',
    'Sortino Ratio',
    'Retorno Anual',
    'Volatilidad Anual',
    'Volatilidad Downside',
    'Exceso de Retorno',
    'Tasa Libre de Riesgo'
])

print("RATIOS DE RIESGO-RETORNO")
display(df_ratios)

RATIOS DE RIESGO-RETORNO


,Valor Formateado
Sharpe Ratio,0.592
Sortino Ratio,0.769
Retorno Anual,31.09%
Volatilidad Anual,44.90%
Volatilidad Downside,34.56%
Exceso de Retorno,26.59%
Tasa Libre de Riesgo,4.50%


In [6]:
dist_results = dist_analyzer.analyze(returns, WEIGHTS)

print("ESTADÍSTICAS BÁSICAS")
print(f"  Media:              {dist_results['mean']:>10.4f} ({dist_results['mean']*100:>7.2f}%)")
print(f"  Mediana:            {dist_results['median']:>10.4f} ({dist_results['median']*100:>7.2f}%)")
print(f"  Desv. Estándar:     {dist_results['std']:>10.4f} ({dist_results['std']*100:>7.2f}%)")
print(f"  Skewness:           {dist_results['skewness']:>10.3f}")
print(f"  Excess Kurtosis:    {dist_results['excess_kurtosis']:>10.3f}")

# Interpretación usando umbrales de config
skew = dist_results['skewness']
if skew > SKEW_THRESH['positive']:
    skew_interp = "Asimetría positiva (cola derecha)"
elif skew < SKEW_THRESH['negative']:
    skew_interp = "Asimetría negativa (cola izquierda)"
else:
    skew_interp = "Aproximadamente simétrica"
print(f"  Interpretación:     {skew_interp}")

# Interpretación kurtosis usando config
kurt = dist_results['excess_kurtosis']
if kurt > KURT_THRESH['positive']:
    kurt_interp = "Leptocúrtica (colas pesadas)"
elif kurt < KURT_THRESH['positive']:
    kurt_interp = "Platicúrtica (colas ligeras)"
else:
    kurt_interp = "Mesocúrtica (normal)"
print(f"  Interpretación:     {kurt_interp}")

print("TEST DE NORMALIDAD (Jarque-Bera)")
print(f"  Estadístico JB:     {dist_results['jb_statistic']:>10.3f}")
print(f"  p-value:            {dist_results['jb_p_value']:>10.4f}")
normal_icon = "Sí" if dist_results['is_normal'] else "No"
print(f"  ¿Distribución Normal?  {normal_icon}")

print("PERCENTILES")
print(f"  1%:                 {dist_results['percentile_1']*100:>10.2f}%")
print(f"  5%:                 {dist_results['percentile_5']*100:>10.2f}%")
print(f"  95%:                {dist_results['percentile_95']*100:>10.2f}%")
print(f"  99%:                {dist_results['percentile_99']*100:>10.2f}%")

ESTADÍSTICAS BÁSICAS
  Media:                  0.0012 (   0.12%)
  Mediana:                0.0011 (   0.11%)
  Desv. Estándar:         0.0283 (   2.83%)
  Skewness:               -0.297
  Excess Kurtosis:        18.083
  Interpretación:     Aproximadamente simétrica
  Interpretación:     Leptocúrtica (colas pesadas)
TEST DE NORMALIDAD (Jarque-Bera)
  Estadístico JB:      16985.601
  p-value:                0.0000
  ¿Distribución Normal?  No
PERCENTILES
  1%:                      -6.43%
  5%:                      -4.02%
  95%:                      4.01%
  99%:                      7.27%


In [7]:
multi_results = var_es_analyzer.calculate_multi_level(
    returns, 
    WEIGHTS, 
    confidence_levels=(0.95, 0.99)
)

for conf_level, methods_dict in multi_results.items():
    print(f"NIVEL DE CONFIANZA: {conf_level*100:.0f}%")
    rows = []
    for method, metrics in methods_dict.items():
        rows.append({
            'Método': method.capitalize(),
            'VaR Diario': f"{metrics['var_daily_pct']:>7.2f}%",
            'ES Diario': f"{metrics['es_daily_pct']:>7.2f}%",
            'VaR Anual': f"{metrics['var_annual_pct']:>7.2f}%",
            'ES Anual': f"{metrics['es_annual_pct']:>7.2f}%"
        })
    
    df = pd.DataFrame(rows)
    display(df)

NIVEL DE CONFIANZA: 95%


,Método,VaR Diario,ES Diario,VaR Anual,ES Anual
0,Historical,-4.02%,-6.15%,-63.83%,-97.67%
1,Parametric,-4.53%,-5.71%,-71.89%,-90.65%
2,Monte_carlo,-4.61%,-5.79%,-73.20%,-91.98%


NIVEL DE CONFIANZA: 99%


,Método,VaR Diario,ES Diario,VaR Anual,ES Anual
0,Historical,-6.43%,-11.21%,-102.14%,-177.91%
1,Parametric,-6.46%,-7.41%,-102.49%,-117.70%
2,Monte_carlo,-6.59%,-7.58%,-104.62%,-120.39%
